In [13]:
import ms3
import pandas as pd
import os
import glob

# Define the base path to the cloned repository
REPO_PATH = "monteverdi_madrigals"

def load_madrigal_data(book_target=None):
    """
    Loads TSV files for the madrigals. 
    If book_target is specified (e.g., 5), it filters for that specific book.
    Returns a dictionary of dictionaries containing notes, harmonies, measures, and chords.
    """
    corpus_data = {
        'notes': {},
        'harmonies': {},
        'measures': {},
        'chords': {}
    }
    
    for data_type in corpus_data.keys():
        # Construct search pattern. If book_target is 5, it searches for "5-*.tsv"
        prefix = f"{book_target}-" if book_target else ""
        search_pattern = os.path.join(REPO_PATH, data_type, f"{prefix}*.{data_type}.tsv")
        
        file_paths = glob.glob(search_pattern)
        
        for file in file_paths:
            # Extract the identifier, e.g., '5-1' or '2-12'
            filename = os.path.basename(file)
            madrigal_id = filename.split('.')[0]
            
            # Load the TSV using the ms3 library
            df = ms3.load_tsv(file)
            corpus_data[data_type][madrigal_id] = df
            
    return corpus_data

In [14]:
# 1. Load the entire corpus of madrigals (Books 1-9 depending on repo contents)
print("Loading the complete Monteverdi corpus...")
all_madrigals = load_madrigal_data()
print(f"Total notes dataframes loaded: {len(all_madrigals['notes'])}")

# 2. Specifically isolate Book 5 for Seconda Pratica / Sprezzatura analysis
print("\nIsolating Book 5 (Seconda Pratica)...")
book_5_data = load_madrigal_data(book_target=5)

# Verify the extraction of Book 5
book_5_keys = sorted(list(book_5_data['notes'].keys()))
print(f"Book 5 madrigals successfully loaded: {book_5_keys}")

# Display a preview of the notes for Madrigal 5-1 (Cruda Amarilli)
print("\nPreview of Book 5, Madrigal 1 (Cruda Amarilli) Notes:")
display(book_5_data['notes']['5-01'].head())

Loading the complete Monteverdi corpus...
Total notes dataframes loaded: 19

Isolating Book 5 (Seconda Pratica)...
Book 5 madrigals successfully loaded: ['5-01', '5-03', '5-04a', '5-04c', '5-04d', '5-04e', '5-05b', '5-05c', '5-08', '5-09', '5-11']

Preview of Book 5, Madrigal 1 (Cruda Amarilli) Notes:


,mc,mn,quarterbeats,quarterbeats_all_endings,duration_qb,mc_onset,mn_onset,timesig,staff,voice,duration,nominal_duration,scalar,tied,tpc,midi,name,octave,chord_id
0,1,1,0,0,4.0,0,0,4/4,5,1,1,1,1,<NA>,1,55,G3,3,4
1,1,1,0,0,4.0,0,0,4/4,4,1,1,1,1,<NA>,2,62,D4,4,3
2,1,1,0,0,4.0,0,0,4/4,3,1,1,1,1,<NA>,1,67,G4,4,2
3,1,1,0,0,4.0,0,0,4/4,2,1,1,1,1,<NA>,5,71,B4,4,1
4,1,1,0,0,4.0,0,0,4/4,1,1,1,1,1,1,2,74,D5,5,0


In [15]:
def extract_tactus_framework_with_offsets(notes_df):
    """
    Separates the bass and vocal lines while calculating both local measure 
    offsets and absolute general offsets for the entire piece.
    """
    # Create a copy to avoid SettingWithCopy warnings
    df = notes_df.copy()
    
    # 1. Calculate 'offset_from_bar'
    # In ms3, 'mc' (metrical position) represents the offset from the start of the measure.
    # We convert it to float to ensure we can perform math on it (ms3 sometimes uses Fractions).
    df['offset_from_bar'] = df['mc'].astype(float)
    df['duration_float'] = df['duration'].astype(float)
    
    # 2. Calculate 'general_offset' (Absolute timeline)
    # First, find where the last note ends in each measure to determine measure lengths.
    df['note_end'] = df['offset_from_bar'] + df['duration_float']
    measure_lengths = df.groupby('mn')['note_end'].max().to_dict()
    
    # Next, calculate the absolute start position of each measure
    measure_starts = {}
    current_timeline_position = 0.0
    
    for mn in sorted(df['mn'].dropna().unique()):
        measure_starts[mn] = current_timeline_position
        # Add the length of the current measure to advance the timeline
        current_timeline_position += measure_lengths.get(mn, 4.0) 
        
    # Map these absolute measure start times back to the main dataframe
    df['measure_start_timeline'] = df['mn'].map(measure_starts)
    
    # The general offset is the measure's absolute start time + the note's local offset
    df['general_offset'] = df['measure_start_timeline'] + df['offset_from_bar']
    
    # Clean up intermediate calculation columns
    columns_to_keep = ['mn', 'offset_from_bar', 'general_offset', 'staff', 'duration_float', 'midi', 'name']
    df = df[columns_to_keep].dropna(subset=['mn', 'offset_from_bar', 'staff'])
    
    # 3. Split the data into Bass and Voices
    max_staff = df['staff'].max()
    
    bass_df = df[df['staff'] == max_staff].copy()
    voices_df = df[df['staff'] < max_staff].copy()
    
    # Sort chronologically using our new general timeline
    bass_df = bass_df.sort_values(by=['general_offset'])
    voices_df = voices_df.sort_values(by=['general_offset'])
    
    return bass_df, voices_df

# Run the updated extraction on Cruda Amarilli (5-01)
bass_anchor, vocal_lines = extract_tactus_framework_with_offsets(book_5_data['notes']['5-01'])

print("--- Updated Basso Continuo (with Offsets) ---")
display(bass_anchor.head())

print("\n--- Updated Vocal Lines (with Offsets) ---")
display(vocal_lines.head())

--- Updated Basso Continuo (with Offsets) ---


,mn,offset_from_bar,general_offset,staff,duration_float,midi,name
0,1,1.0,1.0,5,1.0,55,G3
5,2,2.0,4.0,5,0.5,52,E3
10,2,2.0,4.0,5,0.5,47,B2
18,3,3.0,7.5,5,1.0,50,D3
24,4,4.0,12.5,5,1.0,55,G3



--- Updated Vocal Lines (with Offsets) ---


,mn,offset_from_bar,general_offset,staff,duration_float,midi,name
1,1,1.0,1.0,4,1.00,62,D4
2,1,1.0,1.0,3,1.00,67,G4
3,1,1.0,1.0,2,1.00,71,B4
4,1,1.0,1.0,1,1.00,74,D5
17,2,2.0,4.0,1,0.25,71,B4


In [16]:
def find_sprezzatura_displacements_with_offsets(bass_df, voices_df):
    """
    Identifies rhythmic displacements using the calculated offsets,
    finding vocal notes on off-beats against a steady Basso Continuo.
    """
    b_df = bass_df.copy()
    v_df = voices_df.copy()
    
    # Map bass attacks by measure using the local 'offset_from_bar'
    bass_attacks_by_measure = b_df.groupby('mn')['offset_from_bar'].apply(list).to_dict()
    
    displacements = []
    
    for index, row in v_df.iterrows():
        mn = row['mn']
        local_offset = row['offset_from_bar']
        general_offset = row['general_offset']
        staff = row['staff']
        note_name = row['name']
        
        # Check if the metrical position is an off-beat
        is_off_beat = (local_offset % 1 != 0)
        
        if is_off_beat and mn in bass_attacks_by_measure:
            b_attacks = bass_attacks_by_measure[mn]
            
            # If the bass does NOT attack at this exact moment, record the displacement
            if local_offset not in b_attacks:
                displacements.append({
                    'Measure': mn,
                    'General_Offset': general_offset,
                    'Local_Offset': local_offset,
                    'Voice_Staff': staff,
                    'Voice_Note': note_name,
                    'Bass_Attacks_in_Measure': b_attacks
                })
                
    displacements_df = pd.DataFrame(displacements)
    
    if not displacements_df.empty:
        displacements_df = displacements_df.drop_duplicates(subset=['Measure', 'Voice_Staff', 'Local_Offset'])
        
    return displacements_df



In [17]:
def add_lyrics_to_displacements(displacements_df, chords_df):
    """
    Cross-references the rhythmic displacements with the chords TSV 
    to extract the specific lyrics being sung on those notes.
    """
    c_df = chords_df.copy()
    
    # Convert 'mc' to float to match our 'Local_Offset' and rename columns for merging
    c_df['Local_Offset'] = c_df['mc'].astype(float)
    c_df = c_df.rename(columns={'mn': 'Measure', 'staff': 'Voice_Staff'})
    
    # The ms3 library usually stores lyrics in a column named 'lyrics' or 'text'
    lyric_col = 'lyrics' if 'lyrics' in c_df.columns else 'text' if 'text' in c_df.columns else None
    
    if lyric_col:
        # Filter the chords dataframe to just the relevant matching columns
        lyrics_subset = c_df[['Measure', 'Local_Offset', 'Voice_Staff', lyric_col]].dropna(subset=[lyric_col])
        
        # Merge the lyrics into our displacements dataframe
        merged_df = pd.merge(displacements_df, lyrics_subset, on=['Measure', 'Local_Offset', 'Voice_Staff'], how='left')
        
        # Clean up the output column name for readability
        merged_df = merged_df.rename(columns={lyric_col: 'Lyric_Syllable'})
        return merged_df
    else:
        print("Could not find a lyrics column in the chords dataset.")
        return displacements_df

In [18]:
# 1. Run the updated displacement finder
sprezzatura_flags = find_sprezzatura_displacements_with_offsets(bass_anchor, vocal_lines)

# 2. Add the lyrics by passing the matching chords data for '5-01' (Cruda Amarilli)
sprezzatura_with_lyrics = add_lyrics_to_displacements(sprezzatura_flags, book_5_data['chords']['5-01'])

print(f"Found {len(sprezzatura_with_lyrics)} potential instances of sprezzatura.")
print("\n--- Rhythmic Displacements mapped to Lyrics ---")
display(sprezzatura_with_lyrics.head(15))

Could not find a lyrics column in the chords dataset.
Found 0 potential instances of sprezzatura.

--- Rhythmic Displacements mapped to Lyrics ---


""


In [21]:
# Manual map of emotional peaks in "Cruda Amarilli" based on the actual score
# Dictionary: { 'Word/Phrase': (start_measure, end_measure) }
EMOTIONAL_HOTSPOTS = {
    'Cruda (Cruel)': (1, 5),
    'Ahi lasso! (Alas!)': (12, 14),
    'Aspro (Harsh)': (20, 23),
    'Fera (Fierce / Wild)': (27, 30),
    'Amaramente (Bitterly)': (35, 40),
    'Moriro (I will die)': (65, 73) # The final climax of the madrigal
}

def analyze_hotspots_sprezzatura(vocal_lines_df, bass_anchor_df):
    """
    Analyzes rhythmic sprezzatura (displacements) exclusively 
    in the measures where we know the emotional peaks of the text occur.
    """
    # Prepare the vocal lines dataframe using 'offset_from_bar' instead of 'mc'
    v_df = vocal_lines_df.copy()
    v_df['mc_float'] = v_df['offset_from_bar'].astype(float)
    v_df['mn'] = v_df['mn'].astype(int)
    
    # Prepare the bass dataframe (Tactus)
    b_df = bass_anchor_df.copy()
    b_df['mc_float'] = b_df['offset_from_bar'].astype(float)
    b_df['mn'] = b_df['mn'].astype(int)
    
    # Group bass attacks by measure for quick access
    bass_attacks = b_df.groupby('mn')['mc_float'].apply(list).to_dict()
    
    results = []
    
    # Iterate over our emotional map
    for phrase, (start_measure, end_measure) in EMOTIONAL_HOTSPOTS.items():
        # Filter vocal notes that fall within these measures
        hotspot_notes = v_df[(v_df['mn'] >= start_measure) & (v_df['mn'] <= end_measure)]
        
        for _, note in hotspot_notes.iterrows():
            measure = note['mn']
            beat = note['mc_float']
            staff = note['staff']
            
            b_attacks = bass_attacks.get(measure, [])
            
            # Condition 1: Does the voice enter on an off-beat (fraction)?
            is_off_beat = (beat % 1 != 0)
            
            # Condition 2: Is the bass holding steady while the voice moves?
            against_held_bass = (beat not in b_attacks) if b_attacks else False
            
            if is_off_beat and against_held_bass:
                delivery_type = "Sprezzatura (Syncopation against held bass)"
            elif is_off_beat:
                delivery_type = "Off-beat (Bass also moves)"
            else:
                continue # If it enters "A tempo", we ignore it to only see the friction
                
            results.append({
                'Emotional_Phrase': phrase,
                'Measure': measure,
                'Voice_Beat': beat,
                'Voice_Staff': staff,
                'Delivery_Type': delivery_type
            })
            
    return pd.DataFrame(results)

# Run the analysis
hotspot_analysis_df = analyze_hotspots_sprezzatura(vocal_lines, bass_anchor)

print(f"Found {len(hotspot_analysis_df)} rhythmically displaced attacks in the emotional peaks.")
print("\n--- Evidence of Sprezzatura in Key Phrases ---")
display(hotspot_analysis_df.head(25))

Found 0 rhythmically displaced attacks in the emotional peaks.

--- Evidence of Sprezzatura in Key Phrases ---


""
